In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torchaudio
from model.spectttra import SpecTTTra
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from data_process.dataset import SonicDataset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [3]:
hyper_param_5s = [
    { # alpha
        "input_spec_dim": 128,
        "input_temp_dim": 128,
        "embed_dim": 384,
        "f_clip": 1,
        "t_clip": 3,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # beta
        "input_spec_dim": 128,
        "input_temp_dim": 128,
        "embed_dim": 384,
        "f_clip": 3,
        "t_clip": 5,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # gamma
        "input_spec_dim": 128,
        "input_temp_dim": 128,
        "embed_dim": 384,
        "f_clip": 5,
        "t_clip": 7,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    }
]

hyper_param_120s = [
    { # alpha
        "input_spec_dim": 128,
        "input_temp_dim": 3744,
        "embed_dim": 384,
        "f_clip": 1,
        "t_clip": 3,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # beta
        "input_spec_dim": 128,
        "input_temp_dim": 3744,
        "embed_dim": 384,
        "f_clip": 3,
        "t_clip": 5,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # gamma
        "input_spec_dim": 128,
        "input_temp_dim": 3744,
        "embed_dim": 384,
        "f_clip": 5,
        "t_clip": 7,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    }
]

In [4]:
df = pd.read_csv(f"crop_data/crop{5}.csv")
data = SonicDataset(df)

train_loader = DataLoader(data, batch_size=32, shuffle=True, num_workers=2)
batch = next(iter(train_loader))
X = batch["x"]
y = batch["y"]
print(X.shape, y.shape)


/home/bonxom/miniconda3/envs/Project2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/bonxom/miniconda3/envs/Project2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.Size([32, 1, 128, 157]) torch.Size([32])


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10537 entries, 0 to 10536
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   filename   10537 non-null  str  
 1   path       10537 non-null  str  
 2   fake_type  10537 non-null  str  
 3   label      10537 non-null  int64
dtypes: int64(1), str(3)
memory usage: 329.4 KB


In [6]:
train_loss_hist, train_acc_hist, test_acc_hist = [], [], []
num_epoch = 1

In [7]:
def train(len_clip, version = 1):
    def get_model_param(len_clip, version):
        if len_clip == 5:
            return hyper_param_5s[version-1]
        return hyper_param_120s[version-1]

    if len_clip not in [5, 120]:
        raise ValueError("Clip len only 5s or 120s")
    if version not in [1, 2, 3]:
        raise ValueError("Not availible model version")

    df = pd.read_csv(f"crop_data/crop{len_clip}.csv")

    df_train, df_test = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    print(f"Train data: {df_train.shape}")
    print(f"Test_data: {df_test.shape}")

    train_data = SonicDataset(df_train)
    test_data = SonicDataset(df_test)

    # sample = train_data[0]
    # print(sample.keys())
    # print(type(sample["x"]), sample["x"].shape)
    # print(type(sample["y"]), sample["y"])
    # print(type(sample["filename"]), sample["filename"])
    # print(type(train_data))

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

    # Infer real temporal dimension T from data to avoid Conv1d channel mismatch.
    sample_batch = next(iter(train_loader))
    inferred_temp_dim = sample_batch["x"].shape[-1]

    hyper_param = get_model_param(len_clip, version).copy()
    hyper_param["input_temp_dim"] = inferred_temp_dim
    print(f"Using input_temp_dim={inferred_temp_dim}")

    model = SpecTTTra(**hyper_param).to(device)

    loss = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters())

    for epoch in range(num_epoch):
        # train
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_sample = 0

        for batch in train_loader:
            X = batch["x"].to(device)
            y = batch["y"].to(device).float().unsqueeze(1)

            optimizer.zero_grad()

            logits = model(X)
            cur_loss = loss(logits, y)
            cur_loss.backward()
            optimizer.step()

            total_loss += cur_loss.item() * X.size(0)
            total_sample += X.size(0)

            pred = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (pred == y).sum().item()

        epoch_loss = total_loss / total_sample
        epoch_acc = total_correct / total_sample

        # eval
        model.eval()
        test_sample = 0
        test_correct = 0
        with torch.no_grad():
            for batch in test_loader:
                X = batch["x"].to(device)
                y = batch["y"].to(device).float().unsqueeze(1)

                logits = model(X)
                pred = (torch.sigmoid(logits) >= 0.5).float()
                test_sample += X.size(0)
                test_correct += (y == pred).sum().item()

        test_acc = test_correct / test_sample

        train_loss_hist.append(epoch_loss)
        train_acc_hist.append(epoch_acc)
        test_acc_hist.append(test_acc)

        print(f"Epoch [{epoch+1}/{num_epoch}] "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Train Acc: {epoch_acc:.4f} | "
            f"Test Acc: {test_acc:.4f}")

In [8]:
train(5)

Train data: (8429, 4)
Test_data: (2108, 4)


/home/bonxom/miniconda3/envs/Project2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_keys(['x', 'y', 'filename', 'fake_type', 'path'])
<class 'torch.Tensor'> torch.Size([1, 128, 157])
<class 'torch.Tensor'> tensor(0.)
<class 'str'> fake_07314_suno_1.wav
<class 'data_process.dataset.SonicDataset'>


RuntimeError: Given groups=1, weight of size [384, 128, 1], expected input[32, 157, 128] to have 128 channels, but got 157 channels instead